# Email-Based Job Approval - End-to-End Test

This notebook tests the full email approval flow:

1. DS submits a job to DO
2. `notify` service sends DO an email notification with job details
3. DO replies **"approve"** (or "deny reason") via email
4. `email_approve` service detects reply via Gmail Pub/Sub push notifications
5. Job gets approved and executed automatically
6. DS downloads the results

### Prerequisites
- GCP project with **Gmail API** and **Pub/Sub API** enabled
- OAuth credentials JSON (`test_app_credentials.json`)
- DS token (`token_ds.json`)

## Configuration

Fill in your email addresses below.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

DO_EMAIL = "koenlennartvanderveen@gmail.com"
DS_EMAIL = "koen@openmined.org"

# Credential paths (relative to notebooks/)
APP_CREDENTIALS = Path("../credentials/test_app_credentials.json")
TOKEN_DS = Path("../credentials/token_ds.json")

print(f"App credentials: {APP_CREDENTIALS} - {'exists' if APP_CREDENTIALS.exists() else 'MISSING'}")
print(f"DS token: {TOKEN_DS} - {'exists' if TOKEN_DS.exists() else 'MISSING'}")

## Step 1: Create DO Drive token from app credentials

This converts the OAuth app credentials to a user-authorized Drive token.
It will open a browser window for you to authorize.

In [ ]:
import syft_client as sc
do_token_path = str(APP_CREDENTIALS.parent / "token_do_email_test.json")

# do_token_path = sc.credentials_to_token(
#     credentials_path=str(APP_CREDENTIALS),
#     output_path=do_token_path,
#     do_scopes=True
# )
# print(f"DO token saved to: {do_token_path}")

## Step 2: Login as Data Owner and Data Scientist

In [ ]:
do_client = sc.login_do(email=DO_EMAIL, token_path=str(do_token_path))

In [ ]:
ds_client = sc.login_ds(email=DS_EMAIL, token_path=str(TOKEN_DS))

## Step 2a: possibly delete syftboxes

In [ ]:
# do_client.delete_syftbox()
# ds_client.delete_syftbox()

## Step 3: Set up peer relationship

DS requests access to DO, then DO approves.

In [ ]:
if False:
    ds_client.add_peer(DO_EMAIL)
    
    do_client.load_peers()
    do_client.approve_peer_request(DS_EMAIL, peer_must_exist=False)
    
    print("Peer relationship established")

## Step 4: Set up syft-bg

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import syft_bg

In [ ]:
syft_bg.reset()

In [ ]:
syft_bg.init(
    do_email=DO_EMAIL,
    token_path=do_token_path,
    settings={
        "email_approve":
        {
            "gcp_project_id": "syft-client2"
        }
    }
)

In [ ]:
# syft_bg.config

In [ ]:
syft_bg.ensure_running(services=["email_approve", "notify"], restart=True)

In [ ]:
syft_bg.status

In [ ]:
syft_bg.logs(service="email_approve")

In [ ]:
syft_bg.logs(service="notify")

## Step 6: Create a test script

In [ ]:
import tempfile

SCRIPT_DIR = Path(tempfile.mkdtemp(prefix="email_approve_test_"))
script_path = SCRIPT_DIR / "main.py"

script_content = '''import os
import json

result = {"message": "Hello from email-approved job!", "value": 42}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")
'''

script_path.write_text(script_content)
print(f"Script: {script_path}")
print(script_content)

## Step 7: DS submits the job

After submission, the `notify` service will detect the job and send an email to the DO.

In [ ]:
import uuid

JOB_NAME = f"email_test_{uuid.uuid4().hex[:8]}"

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(script_path),
    job_name=JOB_NAME,
    force_submission=True,
)
print(f"Job '{JOB_NAME}' submitted to {DO_EMAIL}")

In [ ]:
# DO syncs to receive the job
do_client.sync()
print(f"DO jobs: {do_client.job_client.jobs}")

In [ ]:
syft_bg.logs("email_approve")

In [ ]:
JOB_NAME

## Step 8: Check your email and reply "approve"

1. Open the **DO's Gmail inbox**
2. Find the job notification email (subject like "New Job: ...")
3. Reply with just the word: **`approve`**
4. The `email_approve` service will detect the reply and execute the job automatically

To reject instead, reply: `deny <reason>`

In [ ]:
print(f"Check {DO_EMAIL}'s inbox and reply 'approve' to the job notification.")

matching = [j for j in do_client.job_client.jobs if j.name == JOB_NAME]
if matching:
    print("job", matching[0].status)
else:
    print("job not visible yet")

## Step 9: DS downloads results

In [ ]:
ds_client.sync()

In [ ]:
ds_job = [j for j in ds_client.job_client.jobs if j.name == JOB_NAME][0]
print(f""""
Job: {ds_job.name}: {ds_job.status}
=== {ds_job.output_paths[0].name} ===
{ds_job.output_paths[0].read_text()}

""")


## Debugging: Service logs

In [ ]:
print("=== email_approve ===")
for line in syft_bg.logs("email_approve", n=30):
    print(line)

print("\n=== notify ===")
for line in syft_bg.logs("notify", n=20):
    print(line)

## Cleanup